# Crossfilter Dashboard (every view filters the others)

**Family:** Interactive, animated & big data &nbsp;|&nbsp; **Library:** Altair (Vega-Lite) &nbsp;|&nbsp; **Level:** Advanced

> *Interactive, animated & big data:* What-if sliders, linked views, animation and million-point rendering. Use these when exploration or motion adds understanding that a static chart cannot.

## Design purpose

Three coordinated views of one sales table: by region, by product and by month. Clicking a region or product, or brushing a range of months, filters the other two views, while gray bars keep the unfiltered totals for context. A headline states the filtered total and its share. Many-dimensional questions get answered without writing a single group-by.

## When to use

- Exploratory analysis of a cube (region × product × time) by business users
- Find which combination drives a change ("is the Q4 jump all Phones in the West?")
- Replace a stack of pivot tables with one interactive page

## Avoid when

- A one-message presentation (state the finding with one static chart)
- More than four or five dimensions (the page becomes a puzzle; use a BI tool)
- Readers on paper or static slides

## Alternatives

- Small multiples
- Highlight table (region × product)
- Stacked area by product

## Design tips

- The PNG shows the default state (no filters); open the HTML to click and brush
- Keep each view's own selection visible (full opacity) and fade its unselected bars, so readers see what is active
- Draw the unfiltered totals in gray behind the filtered bars to preserve context
- Use one color for "filtered amount" everywhere; identity is already given by position

## Example

Simulated monthly sales ($k) for four regions and five product lines over two years, with seasonality, growth and a strong November–December for Phones in the West.

### Setup

In [1]:
import numpy as np
import pandas as pd
import altair as alt

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
           "#e87ba4", "#008300", "#6250d6", "#e34948"]


@alt.theme.register("vizlib", enable=True)
def vizlib_theme():
    return alt.theme.ThemeConfig({
        "config": {
            "background": "#fcfcfb",
            "view": {"stroke": None},
            "range": {"category": PALETTE},
            "font": "Helvetica, Arial, sans-serif",
            "title": {"anchor": "start", "fontSize": 14, "color": "#0b0b0b"},
            "axis": {"gridColor": "#e6e5e1", "domainColor": "#c9c8c3", "tickColor": "#c9c8c3",
                     "labelColor": "#52514e", "titleColor": "#52514e", "titleFontWeight": "normal"},
            "legend": {"labelColor": "#52514e", "titleColor": "#52514e"},
        }
    })

### Data

In [2]:
rng = np.random.default_rng(31)
regions = {"North": 1.0, "South": 0.8, "East": 1.15, "West": 1.3}
products = {"Laptops": 120, "Phones": 160, "Tablets": 60, "Accessories": 45, "Services": 70}
months = pd.date_range("2023-01-01", periods=24, freq="MS")
rows = []
for region, rw in regions.items():
    for product, base in products.items():
        for i, m in enumerate(months):
            season = 1 + 0.25 * (m.month in (11, 12)) + 0.08 * np.sin(2 * np.pi * m.month / 12)
            boost = 1.6 if (region == "West" and product == "Phones" and m.month in (11, 12)) else 1.0
            sales = base * rw * season * boost * (1 + 0.01 * i) * rng.lognormal(0, 0.08)
            rows.append({"region": region, "product": product, "month": m, "sales": round(sales, 1)})
sales = pd.DataFrame(rows)
sales.pivot_table(index="product", columns="region", values="sales", aggfunc="sum").round(0)

region,East,North,South,West
product,,,,
Accessories,1464.0,1250.0,1017.0,1665.0
Laptops,3852.0,3363.0,2673.0,4411.0
Phones,5345.0,4368.0,3723.0,6453.0
Services,2288.0,1968.0,1587.0,2550.0
Tablets,1917.0,1692.0,1306.0,2170.0


### Plot

In [3]:
region_pick = alt.selection_point(fields=["region"], name="region_pick")
product_pick = alt.selection_point(fields=["product"], name="product_pick")
months_brush = alt.selection_interval(encodings=["x"], name="months_brush")
base = alt.Chart(sales)
region_order, product_order = list(regions), list(products)


def bar_view(field, order, own, others, title):
    y = alt.Y(f"{field}:N", sort=order, title=None)
    x = alt.X("sum(sales):Q", title="Sales ($k)")
    back = base.mark_bar(color="#e3e2de", height=20, cornerRadiusEnd=3).encode(y=y, x=x)
    front = base.mark_bar(color=PALETTE[0], height=20, cornerRadiusEnd=3).encode(
        y=y, x=x, opacity=alt.condition(own, alt.value(1), alt.value(0.35)),
        tooltip=[f"{field}:N", alt.Tooltip("sum(sales):Q", title="Filtered sales ($k)", format=",.0f")])
    for sel in others:
        front = front.transform_filter(sel)
    return alt.layer(back, front.add_params(own)).properties(
        width=300, height=170, title=alt.Title(title, fontSize=12))


region_view = bar_view("region", region_order, region_pick, [product_pick, months_brush], "By region (click)")
product_view = bar_view("product", product_order, product_pick, [region_pick, months_brush], "By product (click)")

x = alt.X("yearmonth(month):T", title=None)
y = alt.Y("sum(sales):Q", title="Sales ($k)")
month_view = alt.layer(
    base.mark_bar(color="#e3e2de").encode(x=x, y=y).add_params(months_brush),
    base.transform_filter(region_pick).transform_filter(product_pick).mark_bar(color=PALETTE[0]).encode(
        x=x, y=y, tooltip=[alt.Tooltip("yearmonth(month):T", title="Month"),
                           alt.Tooltip("sum(sales):Q", title="Filtered sales ($k)", format=",.0f")]),
).properties(width=690, height=180, title=alt.Title("By month (drag to brush a range)", fontSize=12))

headline = (base.transform_joinaggregate(total="sum(sales)")
            .transform_filter(region_pick).transform_filter(product_pick).transform_filter(months_brush)
            .transform_aggregate(selected="sum(sales)", total="max(total)")
            .transform_calculate(label="'Selected sales: $' + format(datum.selected / 1000, ',.2f') + 'M  ·  '"
                                       " + format(datum.selected / datum.total, '.0%') + ' of the two-year total'")
            .mark_text(align="left", fontSize=15, fontWeight="bold")
            .encode(x=alt.value(0), y=alt.value(0), text="label:N")
            .properties(width=690, height=12))

chart = alt.vconcat(headline, alt.hconcat(region_view, product_view, spacing=60), month_view, spacing=22).properties(
    title=alt.Title("Sales crossfilter: every view filters the others (simulated)", subtitle=[
        "Click bars (shift-click for several) and brush months; double-click a view to clear it.", " "]))
chart

alt.VConcatChart(...)

## How to read it

Brush November–December on the timeline and the region and product bars shrink to the holiday share. Phones in the West stand out. Click West and the timeline shows its own seasonality against the gray total. The headline always gives the filtered sales and their share of the total.

---
Standalone script: [`crossfilter_dashboard.py`](crossfilter_dashboard.py) &nbsp;·&nbsp; [Catalogue](../../README.md) &nbsp;·&nbsp; [Gallery](../../gallery/index.html)